# 02 · Diffusion — learned augmentation


> **Research prototype — not a medical device.** Nothing produced by these notebooks may be
> used to diagnose, treat, or make any decision about a patient.


## What this stage is for

Two jobs, both configured under `diffusion.use`:

**`synthesis`** — sample new films conditioned on a label vector and add them to
the training split. Label vectors are resampled from the *training* label
distribution conditioned on each finding in turn, so co-occurrence stays
realistic while every finding gets an equal budget. This is aimed squarely at
the rare combinations a 12k subset barely covers.

**`refine`** — SDEdit. Noise a real film to ~25% of the schedule and denoise it
back. Anatomy survives; texture and acquisition characteristics change. It is a
learned augmentation that respects the image prior instead of jittering pixels.

## Honest scoping

This is a **128 px, ~22M parameter** UNet trained for ~30 epochs on ~10k films.
It is not RoentGen and it will not produce diagnostic-quality radiographs. It is
sized to finish on a free T4 in under an hour.

The samples are therefore used **as extra training rows with a reduced loss
weight** (`train.synthetic_weight: 0.5`), never as a replacement for real data.
Whether they help is an empirical question that notebook 04 answers with an
ablation — do not assume they do.

**The generator sees the training split only.** A model that had seen validation
or test films would launder them back into training and every later number would
be meaningless.

In [ ]:
import subprocess, sys
print(sys.version)
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip() or "no GPU reported")
except FileNotFoundError:
    print("nvidia-smi not found - you are on CPU. Runtime > Change runtime type > T4 GPU.")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# --- 1. where results live -------------------------------------------------
# Mounting Drive is strongly recommended: Colab disconnects, and every stage
# here writes a resumable checkpoint. Without Drive you start over.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = '/content/drive/MyDrive/dvlhg'
else:
    RUN_ROOT = '/content/dvlhg'

# --- 2. get the code -------------------------------------------------------
# Pick ONE. 'clone' is easiest once you have pushed this repo to GitHub.
SOURCE = 'clone'        # 'clone' | 'zip' | 'drive'
REPO_URL = 'https://github.com/abelsangeeth/DVL-Hyperparameter-for-Lung-Disease-Diagnosis.git'
ZIP_PATH = '/content/dvl-hypergraph.zip'          # if SOURCE == 'zip'
DRIVE_CODE = '/content/drive/MyDrive/dvl-hypergraph'  # if SOURCE == 'drive'

import os, shutil, subprocess, sys
CODE = '/content/dvl-hypergraph'
if not os.path.exists(CODE):
    if SOURCE == 'clone':
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, CODE], check=True)
    elif SOURCE == 'zip':
        if not os.path.exists(ZIP_PATH):
            from google.colab import files
            up = files.upload()            # choose the zip from scripts/make_colab_zip.py
            ZIP_PATH = '/content/' + next(iter(up))
        shutil.unpack_archive(ZIP_PATH, '/content/')
    elif SOURCE == 'drive':
        shutil.copytree(DRIVE_CODE, CODE)
print('code at', CODE, '| contents:', sorted(os.listdir(CODE))[:8])

# --- 3. dependencies -------------------------------------------------------
# Colab already ships torch/torchvision built for its CUDA - never reinstall them.
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'open_clip_torch>=2.24', 'timm>=0.9.12', 'transformers>=4.35',
                'fastapi', 'uvicorn', 'python-multipart'], check=True)

sys.path.insert(0, os.path.join(CODE, 'src'))
os.chdir(CODE)
os.environ['PYTHONPATH'] = os.path.join(CODE, 'src')
os.environ['RUN_ROOT'] = RUN_ROOT
print('run root ->', RUN_ROOT)

In [ ]:
# Everything below is overridden on the command line, so this cell is the only
# place you need to edit. Values here are tuned for a Colab T4.
CFG = dict(
    subset_size = 12000,     # frontal studies pulled from MIMIC-CXR-JPG
    text_mode   = 'indication',   # see docs/LEAKAGE.md before changing this
    batch_size  = 24,
    epochs      = 12,
    diffusion_epochs = 30,
    synth_per_class  = 500,
)

def dvlhg(command, **overrides):
    """Run a dvlhg subcommand with RUN_ROOT and any overrides applied."""
    import os, shlex, subprocess, sys
    args = [sys.executable, '-m', 'dvlhg.cli', *shlex.split(command)]
    args += ['--set', f"paths.root={os.environ['RUN_ROOT']}"]
    for key, value in overrides.items():
        args += ['--set', f'{key}={value}']
    print('$', ' '.join(args[2:]))
    return subprocess.run(args, check=True)

In [ ]:
SOURCE_DATASET = 'mimic'   # keep this the same across all notebooks

### Train

~35–50 minutes on a T4 at the defaults. Resumable: re-run the cell after a
disconnect and it continues from the last completed epoch.

In [ ]:
dvlhg('diffusion',
      **{'data.source': SOURCE_DATASET,
         'text.mode': CFG['text_mode'],
         'diffusion.train.epochs': CFG['diffusion_epochs']})

### Look at the samples before trusting them

In [ ]:
import os
from IPython.display import Image as ShowImage, display
path = os.path.join(os.environ['RUN_ROOT'], 'reports', 'figures', 'diffusion_samples.png')
display(ShowImage(filename=path))

Judge these on three things, in order:

1. **Anatomy** — two lung fields, a mediastinum, a diaphragm, ribs. If the
   samples are noise, train longer or lower `diffusion.image_size`.
2. **Conditioning** — does the Cardiomegaly row have a visibly wider cardiac
   silhouette than the "no finding" row? If not, raise
   `diffusion.guidance_scale` (try 3.0–4.0).
3. **Diversity** — if every sample in a row looks identical, guidance is too
   high and the model has collapsed onto one mode.

Only the downstream ablation in notebook 04 tells you whether they *help*.

### Generate the synthetic training rows

In [ ]:
dvlhg('synth',
      **{'data.source': SOURCE_DATASET,
         'text.mode': CFG['text_mode'],
         'diffusion.synth_per_class': CFG['synth_per_class']})

In [ ]:
import json, os
stats = json.load(open(os.path.join(os.environ['RUN_ROOT'], 'synth', 'synth_stats.json')))
print(json.dumps(stats, indent=2))